# 文本遮掩稳健性测试（618+双11）
只在 B站、小红书的618或双11冻结测试行中遮掩。

In [2]:
from pathlib import Path
import sys, random, numpy as np, pandas as pd
from scipy.special import expit
from scipy.stats import spearmanr
from threadpoolctl import threadpool_limits
PYTHON_DIR = Path.cwd() if (Path.cwd() / 'analysis_helpers.py').exists() else Path.cwd() / 'python'
if not (PYTHON_DIR / 'analysis_helpers.py').exists():
    raise FileNotFoundError('请从 lasso-1017 或 lasso-1017/python 目录打开本 Notebook。')
sys.path.insert(0, str(PYTHON_DIR.resolve()))
import analysis_helpers as h
import analysis_scope_1017 as scope
MASK_RATE, SEED, MASK_TOKEN = .20, 20261017, '□'


In [3]:
def mask_text(text, row_id):
    text = '' if pd.isna(text) else str(text)
    candidates = [i for i, ch in enumerate(text) if '\u4e00' <= ch <= '\u9fff']
    if not candidates: return text
    n = max(1, round(len(candidates) * MASK_RATE))
    chosen = set(random.Random(f'{SEED}::{row_id}').sample(candidates, n))
    return ''.join(MASK_TOKEN if i in chosen else ch for i, ch in enumerate(text))
def score_texts(name, texts):
    vectorizer, coef, intercept = h.load_scoring_model(name)
    with threadpool_limits(limits=1):
        return np.asarray(expit(vectorizer.transform(texts) @ coef + intercept)).ravel()


In [4]:
eligible, summaries = scope.mask_eligible_ids(), []
for name in ('Y1', 'Y2'):
    pool, plan = h.load_target(name)
    test = pool.set_index('row_id').loc[plan['test_ids']].reset_index()
    test = test.loc[test.row_id.isin(eligible)].copy()
    test['original'] = score_texts(name, test.text.tolist())
    test['masked'] = score_texts(name, [mask_text(t, r) for t, r in zip(test.text, test.row_id)])
    change = (test.original - test.masked).abs()
    rho, _ = spearmanr(test.original, test.masked)
    summaries.append({'目标': name, '618+双11测试行': len(test), '遮掩比例': MASK_RATE, 'Spearman相关': rho, 'MAE': change.mean(), '95%绝对变化': change.quantile(.95)})
display(pd.DataFrame(summaries))


/Users/xuxiaoxue/Documents/Writings/DA-holiday-work/reasons-analysis-of-shopping-and-rules/lasso-1017/python/analysis_scope_1017.py:64: DtypeWarning: Columns (16,19,21,22,27,28,29,30,31,32,33,34,35,36,37,38,39) have mixed types. Specify dtype option on import or set low_memory=False.
  metadata = pd.read_csv(DATA / "02_X匹配数据" / "comment_X_metadata.csv", dtype={"row_id": str})


,目标,618+双11测试行,遮掩比例,Spearman相关,MAE,95%绝对变化
0,Y1,1317,0.2,0.876872,0.003457,0.002685
1,Y2,1321,0.2,0.843966,0.006251,0.005779
